# Phylogenetics

## Imports

The data we're using is from [Phillip Compeau's website](https://compeau.cbd.cmu.edu/) and I highly encourage you all to check it our for more bioinformatics educational content.

We'll be using real SARS-COV2 data taken from real samples from across the pandemic. We'll use phylogenetics techniques and tools to see the virus mutate over time.

---
The `!` tells google colab, or any python notebook, to execute that line in the shell, not as code. This allows us to run bash commands without needing to open a terminal.

In [ ]:
!wget -qO- https://github.com/iqtree/iqtree2/releases/download/v2.3.6/iqtree-2.3.6-Linux-intel.tar.gz --quiet | tar -xzf -
!wget http://compeau.cbd.cmu.edu/wp-content/uploads/2022/07/UK-Genomes.zip --quiet
!unzip -q -o UK-Genomes.zip
!PATH=$PATH:./iqtree-2.3.6-Linux-intel/bin/

In [ ]:
# Some (VERY DUMB) preprocessing to turn IUPAC sequences into ATCG's only.
# https://www.bioinformatics.org/sms/iupac.html for more details on IUPAC characters.
# This code uses a common command line tool, sed, to replace all non-ACTG characters with '-'.
# Bonus questions: Why is this generally not a good idea?
!sed -i '/^>/!s/[^ATCG-]/-/g' UK-Genomes/2020_11_16/2020_11_16_A.fasta

## What is phylogenetics

**Phylogenetics:** Using genetics to study evolutionary history

Relationships among:
- Species
- Individuals
- Genes

## Phylogenetic Trees
Closer together → more genetic overlap

Clade: Common ancestor and all its descendants
- COVID is a clade in the family coronaviridae


## What is a FASTA FILE
A FASTA file is a text-based file format that is commonly used to contain nucleotide or peptide sequences. [Wikipedia gives a good summary.](https://en.wikipedia.org/wiki/FASTA_format) Every header line starts with a `>`.


Let's explore the FASTA file more. Here is a list of command line functions and documentation that can help you answer the questions below.

```
cat # Displays the entire file to standard out (not suggested on big files like fastas)
less -S # Displays only part of a file at a time (won't work in google colab but invaluable in the command line)
head -n <number> / tail # shows only the first / last number of lines. Defaults to 10
grep # Search a file for lines containing a sequence
wc -l # Counts the number of lines in a file
Tip: include a '!' before each command.
```

We can also chain commands together using a 'pipe' `|` operator. This takes the output from one file and sends it to another.

Here's example that takes the input from one file, searches for only the lines containing the phrase `hello world`, and then counts the number of lines.
```
cat my_file.txt | grep "hello world" | wc -l
```

**Questions:**

Use the data from `2021_02_08` for these questions.

1. How many lines are in this file?
2. What are the first 5 nucleotides in the first sequence?
3. Can you find all the sequence headers?
4. How many samples do we have from Wales?

In [ ]:
!cat /content/UK-Genomes/2021_02_08/2021_02_08.fasta | wc -l

200


In [ ]:
!head  /content/UK-Genomes/2021_02_08/2021_02_08.fasta

>England/CAMC-1262A63/2021
...
>England/CAMC-135A5E4/2021
...
>England/RAND-12BF86D/2021
...
>England/RAND-12BF3A8/2021
...
>England/SHEF-10C5BA9/2021
...


In [ ]:
!cat /content/UK-Genomes/2021_02_08/2021_02_08.fasta | grep ">"

>England/CAMC-1262A63/2021
>England/CAMC-135A5E4/2021
>England/RAND-12BF86D/2021
>England/RAND-12BF3A8/2021
>England/SHEF-10C5BA9/2021
>England/MILK-1285F10/2021
>England/MILK-1285DCB/2021
>England/CAMC-1270E75/2021
>England/LOND-128A2F4/2021
>England/LOND-12E6A76/2021
>England/QEUH-126D3C7/2021
>England/ARCH-000143B01/2021
>England/RAND-12BF01A/2021
>Scotland/EDB15050/2021
>England/ALDP-12896EA/2021
>England/WSFT-25D1753/2021
>England/QEUH-1269D72/2021
>Scotland/EDB15759/2021
>England/RAND-12BF51B/2021
>England/MILK-1284DEA/2021
>England/ALDP-1283A11/2021
>England/BRIS-2611EA/2021
>England/ALDP-1289802/2021
>England/ALDP-12895ED/2021
>England/CAMC-127ED10/2021
>England/RAND-12B3E1C/2021
>Northern_Ireland/RAND-12BEFD2/2021
>Wales/ALDP-12941EB/2021
>England/CAMC-126307F/2021
>Scotland/QEUH-12695B6/2021
>England/CAMC-1263868/2021
>Wales/PHWC-4CA8F7/2021
>Scotland/QEUH-1269905/2021
>England/ALDP-12838CC/2021
>England/CAMC-1264102/2021
>England/ALDP-1295162/2021
>England/RAND-12B3EB2/2021


In [ ]:
!cat /content/UK-Genomes/2021_02_08/2021_02_08.fasta | grep "Wales" | wc -l

6


In [ ]:
!cat /content/UK-Genomes/2021_02_08/2021_02_08_A.fasta

Streaming output truncated to the last 5000 lines.
>England/ALDP-12836B3/2021
NNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNAGATCT
GTTCTCTAAACGAACTTTAAAATCTGTGTGGCTGTCACTCGGCTGCATGCTTAGTGCACT
CACGCAGTATAATTAATAACTAATTACTGTCGTTGACAGGACACGAGTAACTCGTCTATC
TTCTGCAGGCTGCTTACGGTTTCGTCCGTGTTGCAGCCGATCATCAGCACATCTAGGTTT
TGTCCGGGTGTGACCGAAAGGTAAGATGGAGAGCCTTGTCCCTGGTTTCAACGAGAAAAC
ACACGTCCAACTCAGTTTGCCTGTTTTACAGGTTCGCGACGTGCTCGTACGTGGCTTTGG
AGACTCCGTGGAGGAGGTCTTATCAGAGGCACGTCAACATCTTAAAGATGGCACTTGTGG
CTTAGTAGAAGTTGAAAAAGGCGTTTTGCCTCAACTTGAACAGCCCTATGTGTTCATCAA
ACGTTCGGATGCTCGAACTGCACCTCATGGTCATGTTATGGTTGAGCTGGTAGCAGAACT
CGAAGGCATTCAGTACGGTCGTAGTGGTGAGACACTTGGTGTCCTTGTCCCTCATGTGGG
CGAAATACCAGTGGCTTACCGCAAGGTTCTTCTTCGTAAGAACGGTAATAAAGGAGCTGG
TGGCCATAGTTACGGCGCCGATCTAAAGTCATTTGACTTAGGCGACGAGCTTGGCACTGA
TCCTTATGAAGATTTTCAAGAAAACTGGAACACTAAACATAGCAGTGGTGTTACCCGTGA
ACTCATGCGTGAGCTTAACGGAGGGGCATACACTCGCTATGTCGATAACAACTTCTGTGG
CCCTGATGGCTACCCTCTTGAGTGCATTAAAGACCTTCTAGCACGTGCTGGTAAAGCTTC
ATGCACT

## How to build phylogenetic trees with small and large parsimony problems and alignment?

### Base Level Algorithm

When approaching phylogenetics problems, one goal is to identify the least possible number of evolutionary changes for a tree. This is referred to as a Most-parsimonious problem - it is the most likely to occur beacuse it has the least number of nucleotide changes.

As a small example, we can look at the following DNA sequences.

Sequence #1 : ACTGTTGCG

Sequence #2 : TCCGTTACG

How many nucleotides are different between Sequence #1 & Sequence #2?

---

We can apply this idea of directly comparing nucleotides to phylogenetic trees. Each value in the tree is a node and represents a nucleotide.

Before, we counted each difference in nucleotides as +1, but in a biological context, the cost associated with a transition vs a transversion is not equal.

Transition = Purine / Pyrimidine remains a Purine / Pyrimidine Purine = A, G; Pyrimidine = C, T Purines have 2 rings, whereas Pyrimidines have 1.

Transversion = Switches from Purine to Pyrimidine, or vice versa. Physically, this differectly affects DNA size and is less likely, so there is a higher cost associated.

Don't worry too much about the biology here - just know that certain changes are associated with certain costs! This is touched upon in BICD100 (Genetics) if you're interested.

If there's no change, then cost = 0 Transition, cost = 1 Transversion, cost = 2

For instance:
```
     Root
    /   \
   X     Y
  / \   / \
 A   G  A  C
 ```

With this idea, we can build a cost table for the different potential values of X. A cost table associates a cost / value to a potential case (such as X = A or X = C).

| X Value | Cost  |
|---------|-------|
| A       | 0 + 1 |
| C       | 2 + 2 |
| G       | 1 + 0 |
| T       | 2 + 2 |

Try building this tree for the values of Y. (Double click the table below to fill it in)

| Y Value | Cost  |
|---------|-------|
| A       |       |
| C       |       |
| G       |       |
| T       |       |


Question: What would be the most parsimonious state for X/Y, based on the cost tables we've built above? (Hint: Most parsimonious means that there's the least number of changes)

Challenge: How would you apply this to determine the root value?

## Multiple Sequence Alignment

Our data contains folders named after dates which correspond to when each set of COVID genomes was sampled. Inside each folder is a `fasta` file, containing the assembled genomes, and an `A.fasta` file which contains the genomes after they've been preprocessed through Multiple Sequence Alignment (MSA).

To download a file open the folder view on the left and right click the file.

To visualize multiple sequence alignment we can use a tool from the National Center for Biotechnology Information (NCBI): https://www.ncbi.nlm.nih.gov/projects/msaviewer/

Try uploading: `2020_11_16_A.fasta` and then click done. You may have to refresh your browser.

**Questions:**

1. All 100 genomes have the same substring of five nucleotides ranging from column 2251 of the alignment to column 2255. What is the substring?
2. What do the red vertical bars signify in the multiple alignment? What about the gray regions? (Hint: You may find [IUPAC notation](https://en.wikipedia.org/wiki/Nucleic_acid_notation) helpful.)
3. Are there any regions of the genome that you find particularly interesting in terms of studying viral variation? Justify your answer.

Try uploading `2020_12_28_A.fasta` instead.

What changed? Why?

## Building Phylogenetic Trees

Building a phylogenetic tree is a difficult task. To guarantee we have the correct solution (in most cases) we have to test every possible tree. In practice, some tools use heuristics to help find a good enough tree.

We'll use a command line tool called IQtree which you can read about [here](https://academic.oup.com/mbe/article/32/1/268/2925592). It uses a combination of random sampling, iterative improvements, and [Maximum Liklihood Estimation](https://en.wikipedia.org/wiki/Maximum_likelihood_estimation) to find a good enough tree quickly.

Try calling iqtree following the template below with the genomes from `2022_03_07`.
```
!iqtree2 -s <input MSA fasta> -seed <any number> -nt AUTO -prefix <output prefix>
```

- `seed` allows random programs to give the same output everytime
- `nt AUTO` tells IQtree to automatically choose input parameters
- `prefix` specifies the prefix for all output files

Remember `iqtree2` is a command line tool. If you're having trouble check out [IQtree's documentation](http://www.iqtree.org/)

In [ ]:
# Your tree building command here:
!/content/iqtree-2.3.6-Linux-intel/bin/iqtree2 -s /content/UK-Genomes/2022_03_07/2022_03_07_A.fasta -seed 1 -nt AUTO

IQ-TREE multicore version 2.3.6 for Linux x86 64-bit built Aug  1 2024
Developed by Bui Quang Minh, Nguyen Lam Tung, Olga Chernomor, Heiko Schmidt,
Dominik Schrempf, Michael Woodhams, Ly Trong Nhan, Thomas Wong

Host:    31db0181078d (AVX2, FMA3, 12 GB RAM)
Command: /content/iqtree-2.3.6-Linux-intel/bin/iqtree2 -s /content/UK-Genomes/2022_03_07/2022_03_07_A.fasta -seed 1 -nt AUTO
Seed:    1 (Using SPRNG - Scalable Parallel Random Number Generator)
Time:    Fri Jan 17 02:31:28 2025
Kernel:  AVX+FMA - auto-detect threads (2 CPU cores detected)

Reading alignment file /content/UK-Genomes/2022_03_07/2022_03_07_A.fasta ... Fasta format detected
Reading fasta file: done in 0.0270043 secs using 83.55% CPU
Alignment most likely contains DNA/RNA sequences
Alignment has 53 sequences with 29897 columns, 423 distinct patterns
51 parsimony-informative, 153 singleton sites, 29693 constant sites
                            Gap/Ambiguity  Composition  p-value
Analyzing sequences: done in 6.5109e-05 se

After building the tree, the output file we actually care about should be called `<output prefix>2022_03_07_A.fasta.treefile`.

We can analyze it using online tools as well. Download the file and upload it to [Interactive Tree of Life](https://itol.embl.de).

For a better view find the option for `Branch Lengths` and set it to `ignore`.

## Applications of Phylogenetics - COVID

Useful for tracking the growth in COVID variants Rapid genome sequencing → determine location someone was infected (based on the COVID clade)

“The first four cases of COVID-19 in New South Wales, Australia, were found to be closely related to the dominant strain of SARS-CoV-2 found in Wuhan, and these first four cases were all in people who had recently returned from traveling in China” (Source)
Phylogenetics was helpful for tracking and tracing COVID origins and limiting travel during the pandemic.
Tree of 10 million COVID sequences from UC Santa Cruz - the largest tree of genomic sequences of a single species ever assembled (Image Source)

## Applications of Phylogenetics at UCSD

**PanMAN Tool** (Pangenome Mutation-annotate Network) - Turakhia lab

- PanMAN is used to analyze and visualize pangenomes which is especially useful for studying the genetic mutations in viruses like COVID and other microbial datasets
    - Composed of mutation-annotated trees called **PanMATs** (Phylogenetic Analysis of Novel Mutations and Transmissions)
    - **Pangenome**: entire set of genes from all strains within a clade


Useful because:
- By annotating mutations on different branches of the tree it is easier to quickly identify and analyze genetic changes which could be affecting its severity, resistance to vaccines, and transmissibility

Exploration of SARS-CoV-2 mutational and evolutionary landscape using PanMAN vs UShER-MAT (Image Source)

- UShER-MAT: another MAT tool, cannot represent complex mutations

## Resources at UCSD

#### Online Resources
[Simple phylogenetic tree explanation](https://evolution.berkeley.edu/evolution-101/the-history-of-life-looking-at-the-patterns/understanding-phylogenies/)

[More in-depth explanation of creating trees in python](https://taylor-lindsay.github.io/phylogenetics/?utm_source=chatgpt.com)

#### Labs at UCSD
**Turakhia Lab**
 - __[Turakhia Lab Website](https://turakhia.ucsd.edu/)__
 - Led by Prof. Yatish Turakhia
 - Affiliated with Electrical and Computer Engineering
 - Develops automated solutions to construct large-scale phylogenies
 - Also studies
    - __Pangenomics__, genetic variation in a species by looking at multiple genomes. Ex. PanMAN (mentioned previously)
    - __Hardware acceleration__, how to improve computing and make it faster
    - __Outbreak analysis__, looking at real-time phylogenies to analyze pandemics

**Mirarab Lab**
 - __[Mirarab Lab Website](http://eceweb.ucsd.edu/~smirarab/)__
 - Led by Prof. Siavash Mirarab
 - Affiliated with Electrical and Computer Engineering
 - Focuses on reconstructing and utilizing phylogenetic trees
 - Also works in metagenomics (genomics of whole communities of microorganisms), HIV, and Multiple Sequence Alignment, of which they’ve developed 2 new methods
 - Also studies
    - __Metagenomics__, studying whole communities of microorganisms, identifying taxonomic composition (how much of each species)
    - __HIV__, specifically transmission network reconstruction
    - __Multiple Sequence Alignment__, of which they've developed two new methods.
